# سبق 05 - ایجنٹک RAG


## ترتیب

یہ نوٹ بک Microsoft Agent Framework کا استعمال کرتے ہوئے Agentic RAG (Retrieval-Augmented Generation) پیٹرن کو ظاہر کرتی ہے۔

**ضروریات:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — آپ کا Azure AI Search سروس اینڈ پوائنٹ
- `AZURE_SEARCH_API_KEY` — آپ کی Azure AI Search API کلید
- ماحول کے متغیرات کے ذریعے ترتیب دیا گیا Azure OpenAI تعیناتی
- Azure CLI میں لاگ ان شدہ (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAG کیا ہے؟

روایتی RAG ایک مقررہ عمل کی پیروی کرتا ہے: دستاویزات حاصل کریں، پھر جواب تیار کریں۔ **Agentic RAG** اس سے آگے بڑھتا ہے اور ایجنٹ کو خود مختاری دیتا ہے کہ وہ فیصلہ کرے کہ **کب** اور **کیسے** معلومات حاصل کی جائیں۔

Agentic RAG کے ساتھ، ایجنٹ کر سکتا ہے:
- **فیصلہ کریں** کہ سوال کا جواب دینے سے پہلے معلومات حاصل کرنا ضروری ہے یا نہیں
- **انتخاب کریں** کہ کون سا ڈیٹا سورس یا ٹول سے پوچھا جائے
- **جانچیں** حاصل کردہ نتائج کو اور اگر پہلی کوشش ناکافی ہو تو دوبارہ معلومات حاصل کریں
- **معلومات کو ملا کر** متعدد حصولی مراحل سے ایک مربوط جواب تیار کریں

یہ ایجنٹ کو زیادہ لچکدار اور درست بناتا ہے بجائے اس کے کہ وہ ایک جامد حاصل کریں-پھر-پیدا کریں عمل کی پیروی کرے۔


## سرچ ٹول بنانا

Agentic RAG میں، بیرونی ڈیٹا ذرائع کو **ٹولز** کے طور پر لپیٹا جاتا ہے جنہیں ایجنٹ ضرورت کے مطابق بلا سکتا ہے۔ اس سے ایجنٹ کو تلاش کو صرف ایک اور عمل کے طور پر لینے کی اجازت ملتی ہے جو وہ انجام دے سکتا ہے، بجائے اس کے کہ یہ ایک لازمی مرحلہ ہو۔

نیچے ہم ایک سفر کے علم کا ڈیٹا بیس تعریف کرتے ہیں اور اسے ایک ٹول کے طور پر ظاہر کرتے ہیں جسے ایجنٹ منزل کی معلومات دیکھنے کے لیے بلا سکتا ہے۔


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG ایجنٹ کی تعمیر

اب ہم ایک ایجنٹ بناتے ہیں جسے ہدایت دی گئی ہے کہ **ہمیشہ جواب دینے سے پہلے معلومات حاصل کرے**۔ ایجنٹ اپنے جوابات کو اپنے تربیتی ڈیٹا پر انحصار کرنے کے بجائے معلوماتی بنیاد میں گراؤنڈ کرنے کے لیے `search_travel_knowledge` ٹول استعمال کرتا ہے۔


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## تکراری بازیافت — میکر-چیکر پیٹرن

ایجنٹک RAG کا ایک اہم فائدہ **تکراری بازیافت** ہے۔ ایجنٹ ابتدائی معلومات کی تصدیق، بہتری، یا توسیع کے لیے متعدد دوروں پر تلاش کر سکتا ہے — بالکل "میکر-چیکر" ورک فلو کی طرح:

1. **میکر قدم**: ایجنٹ ابتدائی معلومات بازیافت کرتا ہے اور ایک جواب کا مسودہ تیار کرتا ہے۔
2. **چیکر قدم**: ایجنٹ تفصیلات کی تصدیق یا خالی جگہیں پُر کرنے کے لیے اضافی بازیافتیں انجام دیتا ہے۔

نیچے، ایجنٹ سے ایک ایسا سوال پوچھا گیا ہے جس کے لیے متعدد مقامات کا موازنہ کرنا ضروری ہے، جس کی وجہ سے ایجنٹ کو کئی بار تلاش کرنے کی ترغیب ملتی ہے۔


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## خلاصہ

اس سبق میں آپ نے سیکھا کہ Microsoft Agent Framework استعمال کرتے ہوئے **Agentic RAG** سسٹم کیسے بنایا جاتا ہے:

- **Agentic RAG** ایجنٹس کو خودمختاری دیتا ہے کہ وہ معلومات کب حاصل کریں، جس سے حصول متحرک ہو جاتا ہے نہ کہ مقررہ۔
- **ٹولز بطور ڈیٹا ذرائع**: خارجی معلوماتی بنیادوں (جیسے Azure AI Search) کو ایسے ٹولز کے طور پر لپیٹا جاتا ہے جنہیں ایجنٹ کال کر سکتا ہے۔
- **تکراری حصول**: maker-checker پیٹرن ایجنٹ کو متعدد حصولی چکر چلانے کی اجازت دیتا ہے — تلاش، تصدیق، اور بہتری — حتمی جواب دینے سے پہلے۔

پیداواری ماحول میں، آپ in-memory `TRAVEL_KNOWLEDGE_BASE` کو حقیقی Azure AI Search انڈیکس سے تبدیل کریں گے تاکہ بڑے پیمانے پر سفر دستاویزات کی بازیابی کی جا سکے۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈسکلیمر**:  
اس دستاویز کا ترجمہ AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کا استعمال کرتے ہوئے کیا گیا ہے۔ اگرچہ ہم درستگی کے لئے کوشاں ہیں، براہ کرم یاد رکھیں کہ خودکار ترجمے میں غلطیاں یا بے دقتیاں ہو سکتی ہیں۔ اصل دستاویز اپنی مادری زبان میں معتبر ذریعہ تصور کی جانی چاہیے۔ اہم معلومات کے لئے پیشہ ور انسانی ترجمہ تجویز کیا جاتا ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم پر نہیں ہوگی۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
